In [20]:
import numpy as np


class NeuralNetwork:
  def __init__(self, layer_sizes):
    """layer_sizes: e.g., [784, 128, 64, 10] for MNIST"""
    self.L = len(layer_sizes) - 1
    self.W , self.b = [] , []

    for i in range(self.L):
      w = np.random.randn(layer_sizes[i + 1], layer_sizes[i]) * np.sqrt(2/layer_sizes[i]) # weights
      self.W.append(w)
      bais = np.zeros((layer_sizes[i + 1],1)) # baises
      self.b.append(bais)

  def relu(self, z): return np.maximum(0,z)
  def relu_deriv(self, z): return (z > 0).astype(float)
  def softmax(self,z):
    e = np.exp(z - np.max(z, axis=0, keepdims=True))
    return e / np.sum(e, axis=0, keepdims=True) # axis = 0 (find max in cols), keedims -> keep dimensional same

  def forward(self, X):
    """Forward pass - save activations for backprop"""
    self.a, self.z = [X], []

    for i in range(self.L):
      z = self.W[i] @ self.a[-1] + self.b[i]
      self.z.append(z)
      a = self.softmax(z) if i == self.L-1 else self.relu(z)
      self.a.append(a)
    return self.a[-1]

  def backward(self, y, lr = 0.01):
    """Backpropagation - compute and apply gradients"""
    m = y.shape[1]
    delta = self.a[-1] - y

    for i in range(self.L-1, -1):
      dW = (1/m) * delta @ self.a[i].T
      db = (1/m) * np.sum(delta, axis = 1, keepdims=True)

      if(i > 0):
        delta = self.W[i].T @ delta * self.relu_deriv(self.z[i-1])

      self.W[i] -= lr * dW
      self.b[i] -= lr * db


# Train on XOR problem
np.random.seed(42)
X = np.random.randn(2, 1000)
y_labels = ((X[0] > 0) ^ (X[1] > 0)).astype(int)
y = np.eye(2)[y_labels].T

nn = NeuralNetwork([2, 16, 8, 2])
for epoch in range(100):
    out = nn.forward(X)
    loss = -np.mean(np.sum(y * np.log(out + 1e-8), axis=0))
    nn.backward(y, lr=0.1)
    if epoch % 20 == 0:
        acc = np.mean(np.argmax(out, axis=0) == y_labels)
        print(f"Epoch {epoch}: Loss={loss:.4f}, Acc={acc:.1%}")



Epoch 0: Loss=0.6717, Acc=55.7%
Epoch 20: Loss=0.6717, Acc=55.7%
Epoch 40: Loss=0.6717, Acc=55.7%
Epoch 60: Loss=0.6717, Acc=55.7%
Epoch 80: Loss=0.6717, Acc=55.7%


Same NN using PyTorch

In [23]:
import torch
import torch.nn as nn

class SimpleNet(nn.Module):

  def __init__(self):
    super().__init__()
    self.layers = nn.Sequential(
        nn.Linear(2, 16),
        nn.ReLU(),
        nn.Linear(16,8),
        nn.ReLU(),
        nn.Linear(8,2)
    )

  def forward(self, x):
    return self.layers(x)

model = SimpleNet()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.01)

X = torch.randn(1000,2)
y = ((X[:,0] > 0) ^ (X[:,1] > 0)).long()

for epoch in range(100):
  out = model(X)
  loss = criterion(out,y)

  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  if epoch % 20 == 0:
        acc = (out.argmax(1) == y).float().mean()
        print(f"Epoch {epoch}: Loss={loss:.4f}, Acc={acc:.1%}")

# PyTorch handles backprop automatically via autograd!
# loss.backward() computes all gradients
# optimizer.step() applies the updates


Epoch 0: Loss=0.6940, Acc=49.5%
Epoch 20: Loss=0.4702, Acc=84.8%
Epoch 40: Loss=0.1881, Acc=96.8%
Epoch 60: Loss=0.0922, Acc=99.2%
Epoch 80: Loss=0.0627, Acc=99.3%
